# 电视剧推荐器（Rotten Tomatoes + Playwright + OpenAI）

## 练习目标（理念）

用 **烂番茄（Rotten Tomatoes）** 页面上的真实节目信息，做一个小型电视推荐工作流：

1. 用 **Playwright** 打开某类别的电视浏览页（页面依赖 JavaScript 渲染）
2. 用 **BeautifulSoup** 解析高评分节目卡片
3. 再访问每个节目详情页，抓取概要与播出网络 / 流媒体平台
4. 把抓到的**事实**交给 **OpenAI**，格式化成干净的 Markdown 推荐列表

给它一个英文类别名即可，例如 `comedy`、`drama`、`sci fi`、`horror`。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 vs 纯 API | Playwright 渲染 + BeautifulSoup 解析 |
| Chat Completions | `openai.chat.completions.create(...)` |
| system / user messages | system 约束「勿编造」，user 塞入抓取到的 JSON 事实 |
| 事实与生成分离 | 评分/提供商/概要来自网页；模型只做排版 |

## 怎么跑

1. 先跑「一次性安装」单元格（或按下一格用 `uv` 安装 Playwright + Chromium）
2. 确认仓库根目录 `.env` 有 `OPENAI_API_KEY`
3. 选择仓库 `.venv` 内核，从上到下运行；若刚装过包，先**重启内核**再导入


## 环境设置

此笔记本**不**假定 Playwright 已预先装好。

你有两种安装方式：

1. **推荐（笔记本内）**：直接运行下面的一次性安装单元格。
2. **终端（仓库根目录）**：

```bash
uv add playwright
uv run playwright install chromium
```

确保 `.env` 里有 OpenAI API 密钥：

```bash
OPENAI_API_KEY=sk-proj-your-key-here
```

然后选择仓库 `.venv` 内核，从上到下运行。若在会话中刚装过依赖，请先**重启内核**，再重新跑导入单元格。


In [1]:
# ========== 一次性安装：Playwright Python 包 + Chromium 浏览器 ==========
# 可安全重复运行：已安装则跳过 pip，仍会确保 Chromium 可用

# importlib.util：用 find_spec 探测当前内核里有没有 playwright 包
import importlib.util
# subprocess：在子进程里跑 pip / playwright install
import subprocess
# sys：取当前 Python 解释器路径 sys.executable，保证装到「这个」内核
import sys

# 若当前环境找不到 playwright，就用 pip 安装
if importlib.util.find_spec("playwright") is None:
    print("Installing playwright package...")
    # 等价于：python -m pip install playwright
    subprocess.check_call([sys.executable, "-m", "pip", "install", "playwright"])
else:
    print("playwright package already installed")

# 无论包是否已在，都确保 Chromium 浏览器二进制已下载（Playwright 驱动需要）
print("Ensuring Chromium browser is installed for Playwright...")
subprocess.check_call([sys.executable, "-m", "playwright", "install", "chromium"])
print("Playwright setup complete. If imports fail, restart the kernel and rerun.")


playwright package already installed
Ensuring Chromium browser is installed for Playwright...
Playwright setup complete. If imports fail, restart the kernel and rerun.


In [2]:
# ========== 导入 + 环境：抓取 / 解析 / OpenAI 展示所需依赖 ==========

# json：把节目事实序列化进 user prompt
import json
# os：环境变量相关（本格主要靠 load_dotenv）
import os
# re：正则清洗标题、抽取百分数与日期
import re
# datetime：解析 Latest Episode 日期、计算距今天数
from datetime import datetime
# urljoin：把相对链接 /tv/... 拼成绝对 URL
from urllib.parse import urljoin

# BeautifulSoup：解析 Playwright 拿到的 HTML
from bs4 import BeautifulSoup
# load_dotenv：从 .env 读入 OPENAI_API_KEY
from dotenv import load_dotenv
# Markdown + display：在笔记本里漂亮展示推荐结果
from IPython.display import Markdown, display
# OpenAI：Chat Completions 客户端
from openai import OpenAI

# Playwright 异步 API：渲染 JS 页面；TimeoutError 单独起别名便于捕获
try:
    from playwright.async_api import TimeoutError as PlaywrightTimeoutError
    from playwright.async_api import async_playwright
except ImportError as exc:
    # 导入失败时给出可操作提示（英文错误文案保持原样，供程序/用户识别）
    raise ImportError(
        "Playwright is not installed in this kernel. Run the setup cell above, restart the kernel, and run this cell again."
    ) from exc

# 加载 .env；override=True 覆盖已有同名环境变量
load_dotenv(override=True)
# 创建全局 OpenAI 客户端（密钥来自环境变量）
openai = OpenAI()

# 本笔记本用来「排版推荐」的模型 id（勿擅自改成别的，除非你有意换模型）
MODEL = "gpt-4.1-mini"


## 工作流程如何运作

烂番茄浏览页大量依赖 **JavaScript**，只用 `requests` 往往拿不到节目列表。因此流程是：

1. **Playwright** 启动无头 Chromium，等待页面渲染
2. 取出 HTML，交给 **BeautifulSoup** 解析
3. 爬虫把每条节目整理成结构化字段（事实，不是模型编的）
4. **OpenAI** 只负责把这些事实排成可读 Markdown，并被 system prompt 约束「缺失就写 Unknown，不要编造」

结构化字段包括：

- `name`：节目名
- `url`：详情页链接
- `critic_score`：批评家评分（番茄指数）
- `audience_score`：观众评分
- `average_rating`：上述可用分数的均值
- `provider_or_network`：播出网络 / 流媒体
- `synopsis`：剧情概要


In [3]:
# ========== 常量：站点根 URL、浏览器 UA、类别名 → URL slug 映射 ==========

# 烂番茄站点根地址；后面所有路径都基于它拼接
ROTTEN_TOMATOES_BASE_URL = "https://www.rottentomatoes.com"

# 请求头：伪装成常见桌面 Chrome，降低被简单反爬拦截的概率
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# 友好类别名（小写英文）→ 浏览页 URL 里的 genres slug
# 例如 "sci fi" / "science fiction" 都映射到 sci_fi
TV_CATEGORY_SLUGS = {
    "action": "action",
    "adventure": "adventure",
    "animation": "animation",
    "anime": "anime",
    "comedy": "comedy",
    "crime": "crime",
    "documentary": "documentary",
    "drama": "drama",
    "fantasy": "fantasy",
    "game show": "game_show",
    "horror": "horror",
    "kids": "kids_family",
    "kids and family": "kids_family",
    "mystery": "mystery_thriller",
    "reality": "reality",
    "romance": "romance",
    "sci fi": "sci_fi",
    "science fiction": "sci_fi",
    "thriller": "mystery_thriller",
}


In [4]:
# ========== URL 构造 + Playwright 取 HTML + 文本/评分小工具 ==========

# 把友好类别名变成烂番茄 TV 浏览页 URL
def tv_category_url(category):
    """Create a Rotten Tomatoes TV browse URL from a friendly category name."""
    # 去首尾空白、转小写、把连字符当成空格，便于查映射表
    normalized = category.strip().lower().replace("-", " ")
    # 命中映射用官方 slug；否则把空格换成下划线当兜底
    slug = TV_CATEGORY_SLUGS.get(normalized, normalized.replace(" ", "_"))
    # genres:{slug}~sort:popular：按类别筛选并按热度排序
    return f"{ROTTEN_TOMATOES_BASE_URL}/browse/tv_series_browse/genres:{slug}~sort:popular"


# 用无头 Chromium 打开 URL，返回解析好的 BeautifulSoup
async def fetch_rendered_soup(url, wait_ms=600, timeout_ms=20_000, wait_for_networkidle=False):
    """Fetch a JavaScript-rendered page with Playwright and return BeautifulSoup."""
    # async_playwright()：异步上下文，退出时自动清理驱动
    async with async_playwright() as p:
        # headless=True：无界面启动 Chromium
        browser = await p.chromium.launch(headless=True)
        # 新页面：带上 UA 与较大视口，减少响应式布局差异
        page = await browser.new_page(user_agent=headers["User-Agent"], viewport={"width": 1400, "height": 1000})
        try:
            # 等到 DOMContentLoaded 即可（不必等全部资源）
            await page.goto(url, wait_until="domcontentloaded", timeout=timeout_ms)
            # 可选：再等一小段 networkidle；超时就忽略，避免卡死
            if wait_for_networkidle:
                try:
                    await page.wait_for_load_state("networkidle", timeout=2_500)
                except PlaywrightTimeoutError:
                    pass
            # 额外固定等待，给前端 JS 一点渲染时间
            await page.wait_for_timeout(wait_ms)
            # 取出最终 HTML 字符串
            html = await page.content()
        finally:
            # 无论成功失败都关掉浏览器，防止泄漏进程
            await browser.close()
    # html.parser：标准库解析器，依赖少
    return BeautifulSoup(html, "html.parser")


# 把空白压成单空格并去首尾；None 当空串
def clean_text(value):
    return re.sub(r"\s+", " ", value or "").strip()


# 从类似 "87%" 的文本里抽出 0–100 的整数；失败返回 None
def parse_percent(value):
    if value is None:
        return None
    match = re.search(r"(\d{1,3})", str(value))
    if not match:
        return None
    percent = int(match.group(1))
    return percent if 0 <= percent <= 100 else None


# 对若干可能为 None 的分数求平均并四舍五入；全空则 None
def average_percent(*scores):
    available_scores = [score for score in scores if score is not None]
    if not available_scores:
        return None
    return round(sum(available_scores) / len(available_scores))


In [5]:
# ========== 浏览页解析：拆标题、解析最新一集日期、抽卡片、去重排序 ==========

# 「最近在播」窗口：距最新一集不超过这么多天，就视为 currently airing
CURRENT_WINDOW_DAYS = 120


# 把 "Show Name: Season 2" 拆成基础名 + 季信息
def split_show_title(title):
    """Split season-qualified titles into base name and optional season context."""
    match = re.match(r"^(.*?):\s*Season\s+(\d+)\s*$", title, flags=re.IGNORECASE)
    if match:
        # group(1)=剧名，group(2)=季号
        base_name = clean_text(match.group(1))
        season_context = f"Season {match.group(2)}"
        return base_name, season_context
    # 非季标题：整段当名字，季上下文为空串
    return title, ""


# 从卡片文本里解析 "Latest Episode: Apr 29" 这类月日
def parse_latest_episode_date(text):
    """Parse month/day strings like 'Apr 29' from browse cards."""
    match = re.search(r"Latest Episode:\s*([A-Za-z]{3}\s+\d{1,2})", text)
    if not match:
        return None

    month_day = match.group(1)
    now = datetime.now()
    # 页面通常不带年份，先用「今年」拼完整日期
    parsed = datetime.strptime(f"{month_day} {now.year}", "%b %d %Y")
    # 如果日期落在遥远的未来，那么它很可能属于去年。
    if (parsed - now).days > 30:
        parsed = parsed.replace(year=now.year - 1)
    return parsed


# 解析单个 <a href="/tv/..."> 卡片链接 → 候选字典；不合规则返回 None
def parse_show_card(link):
    """Parse one Rotten Tomatoes TV listing link into a structured candidate."""
    href = link.get("href", "")
    # 只要电视剧详情路径
    if not href.startswith("/tv/"):
        return None

    # 取链接可见文本，去掉 Watchlist 噪声
    text = clean_text(link.get_text(" ", strip=True)).replace("Watchlist", "").strip()
    if not text:
        return None

    # 前面最多两个百分数：批评家分、观众分
    scores = [parse_percent(score) for score in re.findall(r"\b\d{1,3}%", text)[:2]]
    # 去掉开头的百分数，留下标题段
    title = re.sub(r"^(?:\d{1,3}%\s*){0,2}", "", text).strip()
    # 截掉 "Latest Episode:" 及其后内容
    title = re.split(r"\s+Latest Episode:\s+", title)[0].strip()
    # 去掉末尾 Trailer 字样
    title = re.sub(r"\s+Trailer\s*$", "", title).strip()

    # 过滤「查看全部」这类导航链接
    if not title or title.lower() in {"view all", "view more"}:
        return None

    base_name, season_context = split_show_title(title)
    critic_score = scores[0] if scores else None
    audience_score = scores[1] if len(scores) > 1 else None

    latest_episode = parse_latest_episode_date(text)
    # 文本里出现 next ep → 明确还有下一集
    has_next_episode = "next ep" in text.lower()
    # 距最新一集的天数；没有日期则为 None
    days_since_latest_episode = (datetime.now() - latest_episode).days if latest_episode else None
    # 有下一集，或最近 N 天内有更新 → 视为当前在播
    is_currently_airing = has_next_episode or (
        days_since_latest_episode is not None and days_since_latest_episode <= CURRENT_WINDOW_DAYS
    )

    return {
        "name": title,
        "base_name": base_name,
        "season_context": season_context,
        "url": urljoin(ROTTEN_TOMATOES_BASE_URL, href),
        "critic_score": critic_score,
        "audience_score": audience_score,
        "average_rating": average_percent(critic_score, audience_score),
        "latest_episode": latest_episode.strftime("%Y-%m-%d") if latest_episode else "",
        "has_next_episode": has_next_episode,
        "days_since_latest_episode": days_since_latest_episode,
        "is_currently_airing": is_currently_airing,
    }


# 从浏览页 soup 收集去重后的候选，并按「在播优先 + 评分」排序
def parse_show_candidates(soup):
    """Collect unique TV show candidates from a Rotten Tomatoes browse page."""
    shows = []
    seen_urls = set()

    # CSS：所有指向 /tv/ 的锚点
    for link in soup.select('a[href^="/tv/"]'):
        show = parse_show_card(link)
        # URL 去重，避免同一节目出现多次
        if show and show["url"] not in seen_urls:
            shows.append(show)
            seen_urls.add(show["url"])

    # reverse=True：True > False，分数高者优先
    shows.sort(
        key=lambda show: (
            show.get("is_currently_airing", False),
            show["critic_score"] is not None,
            show["critic_score"] or -1,
            show["audience_score"] or -1,
        ),
        reverse=True,
    )
    return shows


In [6]:
# ========== 详情页抽取：标签之间切片、网络/平台、概要 ==========

# 在整页纯文本里，取 start_label 之后、任一 end_label 之前的片段
def text_between(page_text, start_label, end_labels):
    start = page_text.find(start_label)
    # 找不到起始标签 → 空串
    if start == -1:
        return ""

    # 跳过标签本身，从内容起点开始
    start += len(start_label)
    # 对每个结束标签找位置，丢掉未找到的 -1
    end_positions = [page_text.find(label, start) for label in end_labels]
    end_positions = [position for position in end_positions if position != -1]
    # 取最近的结束位置；都没有就切到文末
    end = min(end_positions) if end_positions else len(page_text)
    return page_text[start:end].strip(" :-")


# 按「系列信息」标签名抽取一段（自动排除自身标签以免误切）
def extract_series_info(page_text, label):
    end_labels = [
        "Creator",
        "Executive Producer",
        "Network",
        "Rating",
        "Genre",
        "Original Language",
        "Release Date",
        "Seasons",
        "Cast & Crew",
    ]
    return clean_text(text_between(page_text, label, [item for item in end_labels if item != label]))


# 优先读 Network 字段；否则从图片 alt / watch 链接猜流媒体名
def extract_provider_or_network(soup, page_text):
    network = extract_series_info(page_text, "Network")
    if network:
        return network

    provider_names = []
    # 图片 alt 里常有平台 logo 名称
    for image in soup.select("img[alt]"):
        alt_text = clean_text(image.get("alt"))
        if alt_text and alt_text.lower() not in {"rotten tomatoes", "fandango"}:
            provider_names.append(alt_text)

    # 含 watch 的链接文本也可能是平台名
    for link in soup.select("a[href*='watch']"):
        link_text = clean_text(link.get_text(" ", strip=True))
        if link_text and link_text.lower() not in {"where to watch", "watchlist"}:
            provider_names.append(link_text)

    if provider_names:
        # dict.fromkeys：保序去重，最多取前 3 个
        return ", ".join(dict.fromkeys(provider_names[:3]))

    return "Unknown"


# 从详情页 soup 抽出 provider_or_network + synopsis
def extract_show_details(soup):
    # 先把整页打成一行纯文本，方便按标签切片
    page_text = clean_text(soup.get_text(" ", strip=True))
    synopsis = extract_series_info(page_text, "Synopsis")

    # 正文没有 Synopsis 时，退回到 meta description / og:description
    if not synopsis:
        description = soup.select_one('meta[name="description"], meta[property="og:description"]')
        synopsis = clean_text(description.get("content")) if description else ""

    return {
        "provider_or_network": extract_provider_or_network(soup, page_text),
        "synopsis": synopsis or "No synopsis found.",
    }


In [7]:
# ========== 排序打分 + 主流程：浏览 → 富化详情 → 去重 → Top-N ==========

# 返回可比较的元组：在播、信息完整度、新旧程度、评分……
def recommendation_quality_score(show):
    """Score recommendations to prefer currently airing, richer records."""
    is_currently_airing = show.get("is_currently_airing", False)
    # 提供商已知（不是空 / Unknown）
    provider_known = show.get("provider_or_network") not in {None, "", "Unknown"}
    synopsis = (show.get("synopsis") or "").lower()
    # 概要存在，且不是抓取失败占位句
    synopsis_known = synopsis not in {"", "no synopsis found."} and not synopsis.startswith("detail page could not be fetched:")
    # 系列级条目（无 Season 上下文）优先于某一季变体
    is_series_level = not show.get("season_context")
    # 对于当前活跃的节目来说，较小的天数值更好。
    recency_rank = -(show.get("days_since_latest_episode") if show.get("days_since_latest_episode") is not None else 10_000)
    return (
        is_currently_airing,
        provider_known,
        synopsis_known,
        recency_rank,
        is_series_level,
        show.get("critic_score") is not None,
        show.get("critic_score") or -1,
        show.get("audience_score") or -1,
    )


# 端到端：按类别抓浏览页 → 批量补详情 → 按剧名去重 → 取前 max_shows
async def find_tv_recommendations(category, max_shows=5):
    """Browse Rotten Tomatoes for highly rated TV shows in a category."""
    category_page_url = tv_category_url(category)
    # 浏览页多等一会、并尝试 networkidle，列表更稳
    category_soup = await fetch_rendered_soup(
        category_page_url,
        wait_ms=900,
        timeout_ms=25_000,
        wait_for_networkidle=True,
    )
    candidates = parse_show_candidates(category_soup)

    if not candidates:
        raise ValueError(f"No Rotten Tomatoes shows found for category: {category}")

    # 保持详细信息抓取范围，以便笔记本运行不会出现挂起的情况。
    pool_size = min(max(max_shows * 3, max_shows), 12)
    enriched = []

    # 只对前 pool_size 个候选抓详情，控制总耗时
    for show in candidates[:pool_size]:
        record = dict(show)
        try:
            detail_soup = await fetch_rendered_soup(
                record["url"],
                wait_ms=400,
                timeout_ms=12_000,
                wait_for_networkidle=False,
            )
            # 合并详情字段到记录
            record.update(extract_show_details(detail_soup))
        except Exception as exc:
            # 单页失败不中断整体：标记 Unknown + 错误说明
            record.update({"provider_or_network": "Unknown", "synopsis": f"Detail page could not be fetched: {exc}"})
        enriched.append(record)

    # 按标准化标题删除重复的季节级别和系列级别变体。
    best_by_title = {}
    for show in enriched:
        key = clean_text(show.get("base_name") or show.get("name", "")).lower()
        if not key:
            continue
        existing = best_by_title.get(key)
        # 同名保留质量分更高的那条
        if existing is None or recommendation_quality_score(show) > recommendation_quality_score(existing):
            best_by_title[key] = show

    deduped = list(best_by_title.values())
    deduped.sort(key=recommendation_quality_score, reverse=True)

    recommendations = deduped[:max_shows]
    # 展示名统一用 base_name（去掉 Season 后缀）
    for show in recommendations:
        show["name"] = show.get("base_name") or show.get("name", "")

    return recommendations


In [8]:
# ========== Prompt + OpenAI 排版 + 笔记本展示入口 ==========

# system prompt：只格式化事实、禁止编造、优先在播剧（英文原文勿改）
tv_recommendation_system_prompt = """
You are a careful TV recommendation assistant.
Format Rotten Tomatoes facts into a polished markdown recommendation list.
Do not invent providers, networks, ratings, or synopsis details. If a field is Unknown, say Unknown.
Prioritize shows that are currently airing or recently active.
Keep each recommendation concise and useful.
"""


# 组装 Chat Completions 的 messages：system + 带 JSON 事实的 user
def messages_for_tv_recommendations(category, shows):
    # indent=2：让事实在 prompt 里更易读
    facts = json.dumps(shows, indent=2)
    # user_prompt：说明类别、排序意图、每条要输出的字段（英文模板保持原样）
    user_prompt = f"""
The user wants highly rated TV show recommendations for this category: {category}

These shows are already ranked to prioritize currently airing or recently active series.
Here are facts scraped from Rotten Tomatoes. Format them as markdown.
For each show, include:
- name
- optional season context (only if present)
- streaming provider or network
- average rating
- short synopsis
- optional latest episode date (only if present)

Facts:
{facts}
"""
    return [
        {"role": "system", "content": tv_recommendation_system_prompt},
        {"role": "user", "content": user_prompt},
    ]


# 抓取事实 → 调模型排版 → 返回 Markdown 字符串
async def recommend_tv_shows(category, max_shows=5):
    shows = await find_tv_recommendations(category, max_shows=max_shows)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages_for_tv_recommendations(category, shows),
    )
    return response.choices[0].message.content


# 笔记本友好入口：拿到 Markdown 后直接 display
async def display_tv_recommendations(category, max_shows=5):
    recommendations = await recommend_tv_shows(category, max_shows=max_shows)
    display(Markdown(recommendations))


## 运行推荐器

把类别改成你想探索的任意**英文**类别名（须能映射到上一格的 `TV_CATEGORY_SLUGS`）。好的起点：

- `comedy`（喜剧）
- `drama`（剧情）
- `sci fi`（科幻）
- `horror`（恐怖）
- `documentary`（纪录片）
- `kids and family`（儿童与家庭）

`max_shows=5` 时浏览器抓取量适中、速度较快；想要更长列表就增大该参数（详情页请求也会变多）。


In [ ]:
# ========== 演示调用：comedy 类别，最多 5 部 ==========
# 在 Jupyter / async 笔记本里可直接 await；改类别或 max_shows 后重跑本格即可

await display_tv_recommendations("comedy", max_shows=5)


## 故障排除

- **`ModuleNotFoundError: No module named 'playwright'`**：先跑安装单元格，或执行 `uv add playwright`，然后**重启内核**再导入。
- **浏览器启动错误**：在仓库根目录执行 `uv run playwright install chromium`。
- **`OPENAI_API_KEY` 相关错误**：确认仓库根目录 `.env` 存在且密钥有效。
- **结果空洞或怪异**：先试简单类别如 `comedy` / `drama`；烂番茄页面结构可能随时间变化。
- **首次运行很慢**：Playwright 要启动 Chromium 并访问多个详情页；调试时可把 `max_shows` 降到 `3`。
